# 📌 Quantization

![Topic](https://img.shields.io/badge/Topic-Quantization-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-July%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **Quantization shrinks a large language model by storing its weights using fewer bits, typically converting 16 or 32-bit floating-point numbers into 8-bit or 4-bit integers.** This makes the model dramatically smaller and faster to run, at the cost of some precision. It's the main reason you can run a 7B or even 70B parameter model on a consumer laptop or GPU instead of needing a data center.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install numpy` |


---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

**Quantization is a model compression technique that converts the weights and activations within a large language model from high-precision values to lower-precision ones, meaning it changes data from a type that can hold more information to one that holds less. Think of it like compressing a high-resolution photo into a smaller JPEG: you lose some fine detail, but the picture is still recognizable and now takes up far less space.**

Why does this matter? LLMs are typically trained in full precision (FP32) or half precision (FP16), and a model with one billion parameters trained in FP16 alone needs about two gigabytes of memory just to store its weights. Multiply that by 7, 70, or 400 billion parameters, and it's easy to see why memory becomes the bottleneck for who can actually run these models.

**There are two broad families of quantization:**
* **Post-training quantization (PTQ):** converts an already-trained model's floating point numbers to lower precision without any retraining, using only a small calibration dataset. (Cheaper and faster)
* **Quantization-aware training (QAT)** simulates quantization during training itself so the model learns to be robust to the resulting noise. (Better Quality)

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Start with a trained model
Weights normally live as 32-bit or 16-bit floating point numbers.
### 2.2 Measure the range of values
The system looks at the minimum and maximum weight values, either globally or using a small calibration dataset, to figure out how to map them onto a smaller number line.
### 2.3 Build a quantization grid and compute a scale factor
The process determines a quantization grid, a smaller set of discrete values the quantized tensor can take, and then maps the original floating-point numbers onto that grid. A commonly used formula divides each value by a scale factor and rounds to the nearest integer, where the scale is set so the largest weight fits inside the target bit-width's range.
### 2.4 Round each weight to the nearest grid point
In a worked INT8 example, the scale factor is computed as the maximum representable integer value multipled by the maximum weight value, and every weight is then multipled by that scale and rounded.
### 2.5 Store the compact integers plus the scale (and sometimes a "zero point")
This pair of numbers is what lets the system reconstruct an approximation of the original value later.
### 2.6 Dequantize at inference time (or compute directly in integer math)
When converting back to the original format, the recovered numbers are close but not identical to the originals, for example a value of roughly 0.5415 might come back as roughly 0.543, which is the rounding error inherent to the quantize-dequantize process.

![Quantization.png](../assets/Quantization.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Massive memory savings** | Quantization can reduce model size by up to 75 to 90 percent while maintaining most of the performance. |
| 🟢 | **Faster inference** | Lower-precision integer math runs faster on most hardware, so a quantized model produces tokens quicker than its full-precision counterpart. |
| 🟢 | **Lower cost and energy use** | Quantization produces LLMs that consume less memory, require less storage space, are more energy-efficient, and are capable of faster inference. |
| 🟢 | **Democratized access** | This dramatic size reduction enables powerful models to run on consumer hardware, reducing costs and democratizing access to AI capabilities. |
| 🟢 | **Enables previously impossible deployments** | Techniques built on quantization, like NF4 and Double Quantization, are what make it feasible to run or even fine-tune a large model on a single GPU. QLoRA, for instance, quantizes a base model's weights to 4-bit specifically to reduce the memory needed to run it on one GPU |
| 🔴 | **Accuracy loss at low bit-widths** | Extreme quantization down to 2 or 3 bits can cause significant performance drops. |
| 🔴 | **Task sensitivity** | Mathematical reasoning and complex logic tasks tend to suffer more from quantization than other tasks. |
| 🔴 | **Smaller models are more fragile** | Smaller models are generally more sensitive to the effects of quantization than larger ones. |
| 🔴 | **Hardware and tooling fragmentation** | Formats like GPTQ, AWQ, GGUF, and bitsandbytes each use different formats and toolchains, so picking the wrong one can cause inference engine incompatibility. |
| 🔴 | **Security blind spots** | Research has shown that current evaluation practices don't fully capture how quantization affects model behavior, and an adversary can build a model that looks safe in full precision but becomes harmful only after it's quantized. |

---
## 4. Code Example

> **Goal:** This is a simplified simulation of symmetric INT8 quantization:

In [1]:
import numpy as np

def quantize(weights, bits=8):
    """Simulate symmetric quantization of a weight array."""
    qmax = 2 ** (bits - 1) - 1          # e.g. 127 for INT8
    scale = np.max(np.abs(weights)) / qmax
    quantized = np.round(weights / scale).astype(np.int8)
    return quantized, scale

def dequantize(quantized, scale):
    """Reconstruct an approximation of the original weights."""
    return quantized.astype(np.float32) * scale

# Example: a tiny "weight matrix" from a toy model
original_weights = np.array([0.5415, -0.932, 0.0609, 0.271], dtype=np.float32)

q_weights, scale = quantize(original_weights, bits=8)
recovered_weights = dequantize(q_weights, scale)

print("Original :", original_weights)
print("Quantized:", q_weights, " (scale =", round(float(scale), 5), ")")
print("Recovered:", recovered_weights)
print("Memory   : 4 bytes/weight -> 1 byte/weight (75% smaller)")

Original : [ 0.5415 -0.932   0.0609  0.271 ]
Quantized: [  74 -127    8   37]  (scale = 0.00734 )
Recovered: [ 0.5430551  -0.932       0.05870866  0.27152756]
Memory   : 4 bytes/weight -> 1 byte/weight (75% smaller)


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **Quantization trades precision for size.** Fewer bits per weight means a smaller, faster model at the cost of some accuracy.
- **PTQ is cheap, QAT is careful.** Post-training quantization is fast and easy; quantization-aware training costs more but preserves quality better.
- **The scale factor is the whole trick.** A single number lets you round floats onto a small integer grid and (approximately) reverse the process later.
- **Not all bits are equal.** Methods like AWQ protect the small number of "important" weights instead of compressing everything equally.
- **The right format depends on your hardware.** GGUF suits CPUs, GPTQ and AWQ target GPUs, and the "best" choice varies by workload.

</div>